# Airport Connectivity Analysis - Assignment 3

This notebook uses reusable utilities from `utils.py` so you can swap datasets by changing only the input paths/schema config.

Visualizations replicate Assignment 2 analyses using **Bokeh** with map-style coordinates, directed arrows, and linked edge highlighting on node hover/select.

In [10]:
from pathlib import Path
import importlib
from IPython.display import display
from bokeh.io import output_notebook, show

import utils
importlib.reload(utils)

from utils import (
    AirportDatasetConfig,
    run_pipeline,
    plot_all_connections,
    plot_one_way_connections,
    plot_one_way_degree_diff,
    plot_two_way_connections,
    plot_centrality_grid,
    plot_degree_histogram,
    plot_local_clustering_map,
    plot_center_periphery,
    plot_degree_distribution_loglog,
    louvain_communities_plot,
    leiden_communities_plot,
    girvan_newman_communities_plot,
    louvain_communities_map_plot,
    plot_louvain_coassignment_heatmaps_pair,
    plot_louvain_community_mean_degrees,
    plot_louvain_community_node_counts,
    plot_louvain_modularity_trajectory,
    louvain_reference_partition,
    louvain_modularity_by_level,
)

output_notebook()

Loading BokehJS ...

In [11]:
# Change only this block to use another airport dataset
DATA_ROOT = Path('../data')

config = AirportDatasetConfig(
    nodes_path=str(DATA_ROOT / 'reachability-meta.csv' / 'reachability-meta.csv'),
    edges_path=str(DATA_ROOT / 'reachability.txt' / 'reachability.txt'),
    node_id_col='node_id',
    node_name_col='name',
    node_lat_col='latitude',
    node_lon_col='longitude',
    node_pop_col='metro_pop',
    edge_source_col='FromNodeId',
    edge_target_col='ToNodeId',
    edge_weight_col='Weight',
)

In [12]:
result = run_pipeline(config)

G = result['G']
G_one_way = result['G_one_way']
G_undirected = result['G_undirected']

print('Directed summary:')
display(result['directed_table'])

print('Undirected summary:')
display(result['undirected_table'])

Directed summary:


,nodes,edges,scc_count,largest_scc_size
0,456,71959,1,456


Undirected summary:


,nodes,edges,avg_degree,density,connected_components,largest_cc_size,avg_clustering,transitivity,mean_shortest_path,diameter,radius
0,456,34012,149.175439,0.327858,1,456,0.806788,0.590841,1.674745,3,2


In [13]:
show(plot_all_connections(G, max_edges=350))
show(plot_one_way_connections(G_one_way, max_edges=600))
show(plot_one_way_degree_diff(G_one_way, max_edges=600))
show(plot_two_way_connections(G_undirected, max_edges=600))

In [14]:
centrality_grid = plot_centrality_grid(G_undirected, max_edges_each=700)
show(centrality_grid)

In [15]:
show(plot_degree_histogram(G_undirected))
show(plot_local_clustering_map(G_undirected, max_edges=900))
show(plot_center_periphery(G_undirected, max_edges=900))
show(plot_degree_distribution_loglog(G_undirected, add_fit=False))

## Community Detection Comparison
This section compares Louvain, Leiden, and Girvan-Newman communities on the undirected airport graph.

In [ ]:
louvain_fig, louvain_modularity = louvain_communities_plot(G_undirected, max_edges=900)
print(f'Louvain modularity: {louvain_modularity:.4f}')
show(louvain_fig)

### Community 1: part of the American Airlines / United / Air Canada regional ecosystems


Louvain modularity: 0.1486


## Louvain community stability

Repeated Louvain runs (co-assignment matrix), mean degree per community, and modularity across hierarchical levels of one run.

In [ ]:
import importlib

importlib.reload(utils)

from utils import (
    louvain_communities_map_plot,
    plot_louvain_coassignment_heatmaps_pair,
    plot_louvain_community_mean_degrees,
    plot_louvain_community_node_counts,
    plot_louvain_modularity_trajectory,
    louvain_reference_partition,
    louvain_pre_final_communities_map_plot,
    louvain_modularity_by_level,
)

N_LOUVAIN_RUNS = 40
LOUVAIN_SEED = 42
HEATMAP_SAMPLE_SEED = 2024
MAX_HEATMAP_NODES = 40

# Reference partition (shared by map, heatmap order, bar colors, hover labels)
ref_communities, ref_modularity, ref_membership = louvain_reference_partition(
    G_undirected,
    seed=LOUVAIN_SEED,
)

# All airports per Louvain community (final partition)
print("Louvain communities (final partition) - airports per community:")
for comm_id, community in enumerate(ref_communities, start=1):
    airport_names = sorted(
        G_undirected.nodes[n].get("name", str(n)) for n in community
    )
    print(f"\nCommunity {comm_id} ({len(airport_names)} airports):")
    for name in airport_names:
        print(f"  - {name}")

# 0) Community map — use this to read Community 1, 2, … in the bar chart
map_fig, _, _, _ = louvain_communities_map_plot(
    G_undirected,
    communities=ref_communities,
    modularity_score=ref_modularity,
    seed=LOUVAIN_SEED,
    max_edges=900,
)
show(map_fig)
print(f"Reference Louvain modularity (final): {ref_modularity:.4f}")


# 1) Co-assignment heatmaps side by side (hover highlights row/column)
co_layout, co_matrices, co_label_groups, co_selected = plot_louvain_coassignment_heatmaps_pair(
    G_undirected,
    communities=ref_communities,
    membership=ref_membership,
    n_runs=N_LOUVAIN_RUNS,
    seed=LOUVAIN_SEED,
    max_nodes=MAX_HEATMAP_NODES,
    heatmap_sample_seed=HEATMAP_SAMPLE_SEED,
)
show(co_layout)
print(
    f"Left: {len(co_label_groups[0])} random high-degree airports | "
    f"Right: {len(co_label_groups[1])} balanced per community "
    f"({G_undirected.number_of_nodes()} nodes in graph)."
)


# 1b) Co-assignment heatmaps side by side without node limit
co_layout, co_matrices, co_label_groups, co_selected = plot_louvain_coassignment_heatmaps_pair(
    G_undirected,
    communities=ref_communities,
    membership=ref_membership,
    n_runs=N_LOUVAIN_RUNS,
    seed=LOUVAIN_SEED,
    max_nodes=10000,
    heatmap_sample_seed=HEATMAP_SAMPLE_SEED,
)
show(co_layout)
print(
    f"Left: {len(co_label_groups[0])} random high-degree airports | "
    f"Right: {len(co_label_groups[1])} balanced per community "
    f"({G_undirected.number_of_nodes()} nodes in graph)."
)


Louvain communities (final partition) - airports per community:

Community 1 (134 airports):
  - Abbotsford, BC
  - Albany, NY
  - Altoona, PA
  - Augusta, ME
  - Bagotville, QC
  - Baie Comeau, QC
  - Bar Harbor, ME
  - Bathurst, NB
  - Beckley, WV
  - Blanc Sablon, QC
  - Bluefield, WV
  - Boston, MA
  - Bradford, PA
  - Burlington, IA
  - Campbell River, BC
  - Cape Girardeau, MO
  - Castlegar, BC
  - Charlottetown, PE
  - Chibougamau, QC
  - Chicago, IL
  - Clarksburg, WV
  - Cleveland, OH
  - Columbia, MO
  - Comox, BC
  - Cranbrook, BC
  - Dallas/Fort Worth, TX
  - Decatur, IL
  - Deer Lake, NL
  - Denver, CO
  - Detroit, MI
  - Dubois, PA
  - Escanaba, MI
  - Franklin, PA
  - Fredericton, NB
  - Ft. Huachuca/Sierra Vista, AZ
  - Ft. Lauderdale, FL
  - Ft. Leonard Wood, MO
  - Ft. McMurray, AB
  - Ft. St John, BC
  - Gaspe, QC
  - Goose Bay, NL
  - Grande Prairie, AB
  - Great Bend, KS
  - Hagerstown, MD
  - Halifax, NS
  - Hartford, CT
  - Iles de la Madeleine, QC
  - Iron Mount

Reference Louvain modularity (final): 0.1486


Left: 40 random high-degree airports | Right: 40 balanced per community (456 nodes in graph).


Left: 456 random high-degree airports | Right: 140 balanced per community (456 nodes in graph).


In [19]:
from pathlib import Path

from bokeh.io import show

from utils import (
    load_world_edge_list,
    build_world_graphs,
    plot_world_directed_overview,
    plot_world_undirected_overview,
    plot_world_local_clustering,
    plot_world_center_periphery,
    compute_world_center_periphery,
    plot_world_degree_histogram,
    plot_world_degree_distribution_loglog,
    plot_world_louvain_communities,
    louvain_world_coassignment,
 )

# World airport graph (no coordinates available)
WORLD_EDGES_PATH = Path("../data/download.tsv.maayan-faa/maayan-faa/out.maayan-faa")

world_edges = load_world_edge_list(WORLD_EDGES_PATH)
G_world_directed, G_world_undirected = build_world_graphs(world_edges)

print(
    f"World graph - directed: {G_world_directed.number_of_nodes()} nodes, "
    f"{G_world_directed.number_of_edges()} edges"
 )
print(
    f"World graph - undirected: {G_world_undirected.number_of_nodes()} nodes, "
    f"{G_world_undirected.number_of_edges()} edges"
 )

# Exploratory plots (non-map layout)
show(plot_world_directed_overview(G_world_directed, max_edges=2000, seed=42))
show(plot_world_undirected_overview(G_world_undirected, max_edges=2000, seed=42))

# Degree histogram and distribution
show(plot_world_degree_histogram(G_world_undirected, bins=30))
show(plot_world_degree_distribution_loglog(G_world_undirected))

# Local clustering coefficient per node
show(plot_world_local_clustering(G_world_undirected, max_edges=2000, seed=42))

# Center and periphery (largest connected component)
show(plot_world_center_periphery(G_world_undirected, max_edges=2000, seed=42))
center_stats = compute_world_center_periphery(G_world_undirected)
print(
    f"\nLargest connected component: {center_stats['lcc_nodes']} nodes, "
    f"{center_stats['lcc_edges']} edges"
 )
print(f"Radius: {center_stats['radius']} | Diameter: {center_stats['diameter']}")
print(
    f"Center nodes: {len(center_stats['center_nodes'])} | "
    f"Periphery nodes: {len(center_stats['periphery_nodes'])}"
 )

# Louvain communities by color (non-map)
show(plot_world_louvain_communities(G_world_undirected, max_edges=2000, seed=42))

# Louvain communities and co-assignment matrix
communities, modularity_score, co_layout, co_label_groups = louvain_world_coassignment(
    G_world_undirected,
    n_runs=40,
    seed=42,
    max_nodes=40,
    heatmap_sample_seed=2024,
 )

community_sizes = sorted([len(c) for c in communities], reverse=True)
print(f"\nLouvain modularity (world graph): {modularity_score:.4f}")
print(f"Louvain communities: {len(communities)}")
print(f"Top 10 community sizes: {community_sizes[:10]}")
show(co_layout)
print(
    f"Co-assignment heatmaps: {len(co_label_groups[0])} random high-degree nodes | "
    f"{len(co_label_groups[1])} balanced per community"
 )

# Co-assignment heatmaps without node limit
communities_all, modularity_score_all, co_layout_all, co_label_groups_all = louvain_world_coassignment(
    G_world_undirected,
    n_runs=40,
    seed=42,
    max_nodes=10000,
    heatmap_sample_seed=2024,
 )
show(co_layout_all)
print(
    f"Co-assignment heatmaps (all nodes): {len(co_label_groups_all[0])} random high-degree nodes | "
    f"{len(co_label_groups_all[1])} balanced per community"
 )

World graph - directed: 1226 nodes, 2615 edges
World graph - undirected: 1226 nodes, 2408 edges



Largest connected component: 1226 nodes, 2408 edges
Radius: 9 | Diameter: 17
Center nodes: 1 | Periphery nodes: 8


Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py", line 3526, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\carlo\AppData\Local\Temp\ipykernel_32036\1965343501.py", line 62, in <module>
    communities, modularity_score, co_layout, co_label_groups = louvain_world_coassignment(
                                                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\carlo\Documents\uni\master_data_science\sem2\complex_networks\git_repository\Complex-Networks\aeropuertos_EEUU\assignment_3_community_analisis_v3_final\utils.py", line 865, in louvain_world_coassignment
    plot = _base_layout_figure("Louvain Communities (World)", pos)
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\carlo\Documents\uni\master_data_science\sem2\complex_networks\git_repository\Complex-Networks\aeropuertos_EEUU\assignment_3_community_analisis_v3_final\